In [ ]:
# train_network.ipynb
#
# Trains a neural network to replace the BGK collision operator.
#
# The network takes f_pre (9 pre-collision populations) as input and predicts
# f_post (9 post-collision populations).  Three design choices enforce physical
# correctness by construction:
#
#   1. Density normalisation  — inputs and outputs are divided by rho so they
#      sum to 1 (probability distributions).  The network learns the *shape*
#      of f, independent of the density scale.
#
#   2. D4 equivariance  — the model applies the same sub-network to all 8
#      rotated/mirrored copies of the input, then averages the back-transformed
#      outputs.  This guarantees the collision operator respects the square-
#      lattice symmetry without any extra training data or regularisation.
#
#   3. AlgReconstruction  — after each sub-network prediction, 3 of the 9
#      output components are overwritten using the conservation laws (mass +
#      2 momentum).  This enforces exact conservation regardless of what the
#      network predicted for those directions.

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import train_test_split

from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras import backend as K

import datetime
import numpy as np
from utils import *

In [ ]:
# float64 precision is needed because the non-equilibrium part f_neq is several
# orders of magnitude smaller than f_eq.  In float32 the small differences
# between f_pre and f_eq would be lost to rounding error.
K.set_floatx('float64')

In [ ]:
def load_data(fname):
    """Load the dataset produced by create_trainset.ipynb.

    Returns
    -------
    feq   : (N, 9) — equilibrium populations, needed by AlgReconstruction
    fpre  : (N, 9) — pre-collision populations  (network input)
    fpost : (N, 9) — post-collision populations (network label)
    """
    data = np.load(fname, allow_pickle=True)

    feq   = data['f_eq']
    fpre  = data['f_pre']
    fpost = data['f_post']

    return feq, fpre, fpost

In [ ]:
def sequential_model(Q=9, n_hidden_layers=2, n_per_layer=50, activation="relu",
                     ll_activation="linear", bias=False):
    """Build a simple fully-connected (MLP) sub-network.

    This is the shared-weight backbone that is applied to each of the 8
    symmetry-transformed copies of the input inside create_model.

    Parameters
    ----------
    Q               : input/output size (9 for D2Q9)
    n_hidden_layers : number of hidden Dense layers
    n_per_layer     : neurons per hidden layer
    activation      : activation function for hidden layers
    ll_activation   : activation function for the output layer.
                      "softmax" ensures all 9 outputs are positive and sum to 1,
                      matching the density-normalised input convention.
    bias            : whether to include bias terms (False enforces f=0 → output=0)
    """
    model = Sequential([
        keras.Input(shape=(Q,)),
        Dense(n_per_layer, activation=activation, use_bias=bias, kernel_initializer="he_uniform"),
    ])

    for jj in range(n_hidden_layers):
        model.add(Dense(n_per_layer, activation=activation, use_bias=bias, kernel_initializer="he_uniform"))

    # Output layer: one value per lattice direction.
    # softmax guarantees positivity and unit sum — physically valid populations.
    model.add(Dense(Q, activation=ll_activation, use_bias=bias, kernel_initializer="he_uniform"))

    return model


def create_model(loss="mape", optimizer="adam", Q=9,
                 n_hidden_layers=2, n_per_layer=50, activation="relu",
                 ll_activation="linear", bias=False):
    """Build the full equivariant collision-operator model.

    Architecture (D4-equivariant group-averaging):
    -----------------------------------------------
    1. D4Symmetry     — lift the input to all 8 symmetry-transformed copies
    2. seq_model      — apply the same MLP to each copy (shared weights)
    3. AlgReconstruction — enforce conservation laws on each copy's output
    4. D4AntiSymmetry — undo each transform, bringing all copies back to the
                        original orientation
    5. Average        — average the 8 outputs into one prediction

    Because steps 1–4 are equivariant by construction, the averaged output is
    guaranteed to be symmetric under all D4 transforms — no data augmentation
    needed.

    Parameters
    ----------
    loss         : training loss (use rmsre from utils for scale-invariant error)
    ll_activation: output activation; "softmax" enforces positivity + unit sum
    """
    the_input = keras.Input(shape=(Q,))  # f_pre, density-normalised, shape (batch, 9)

    # Single MLP sub-network — weights are shared across all 8 symmetry copies
    seq_model = sequential_model(Q, n_hidden_layers, n_per_layer,
                                 ll_activation, ll_activation, bias)

    # Step 1 — produce 8 symmetry-transformed versions of f_pre
    input_lst = D4Symmetry()(the_input)  # list of 8 tensors, each (batch, 9)

    # Step 2 — run each transformed input through the shared MLP
    output_lst = [seq_model(x) for k, x in enumerate(input_lst)]

    # Step 3 — fix the 3 conservation-constrained directions in each output
    # AlgReconstruction uses the corresponding transformed f_pre as reference
    output_lst = [AlgReconstruction()(input_lst[k], x) for k, x in enumerate(output_lst)]

    # Step 4 — rotate/mirror each output back to the canonical orientation
    output_lst = D4AntiSymmetry()(output_lst)

    # Step 5 — average the 8 now-aligned predictions into one output
    the_output = layers.Average()(output_lst)  # shape (batch, 9)

    model = keras.Model(inputs=the_input, outputs=the_output)
    model.compile(loss=loss, optimizer=optimizer)

    return model

In [15]:
###########################################################
# lattice velocities and weights
# Needed if you want to use AlgReconstructionSymm
# c, _, _, _ = LB_stencil()
# c = np.array(c, dtype=float)

In [ ]:
# Load dataset
feq, fpre, fpost = load_data('example_dataset.npz')

# Normalise by density (rho = sum of all 9 populations).
# After this, each sample sums to exactly 1 — it becomes a probability
# distribution over the 9 directions.  This does two things:
#   - Removes the density scale so the network learns the *shape* of f, not its magnitude.
#   - Matches the softmax output, which also sums to 1, making input and output
#     live in the same space.
feq   = feq   / np.sum(feq,   axis=1)[:, np.newaxis]  # shape (N, 9), each row sums to 1
fpre  = fpre  / np.sum(fpre,  axis=1)[:, np.newaxis]
fpost = fpost / np.sum(fpost, axis=1)[:, np.newaxis]

# Split into training (70%) and validation (30%) sets.
# Only fpre and fpost are needed here; feq is used inside AlgReconstruction
# which is baked into the model and receives its input from the D4Symmetry layer.
fpre_train, fpre_test, fpost_train, fpost_test = train_test_split(
    fpre, fpost, test_size=0.3, shuffle=True
)

In [ ]:
# -----------------------------------------------------------------------
# Hyperparameters
# -----------------------------------------------------------------------
batch_size = 32    # samples per gradient update
n_epochs   = 200   # maximum training epochs (early stopping will cut this short)
patience   = 50    # stop if validation loss does not improve for this many epochs
verbose    = 1

# Build model.
# loss=rmsre : Root Mean Square Relative Error (defined in utils.py).
#              Scale-invariant — penalises relative rather than absolute errors,
#              so small populations (diagonal directions) are weighted fairly.
# ll_activation="softmax" : output sums to 1 and is strictly positive,
#              consistent with the density-normalised inputs.
model = create_model(loss=rmsre, ll_activation="softmax")

# -----------------------------------------------------------------------
# Callbacks
# -----------------------------------------------------------------------

# Stop training early if validation loss stops improving, then restore the
# best weights seen during training (avoids overfitting to the training set).
es_callback = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

# Independently checkpoint the best model to disk so it survives a crash.
ck_callback = ModelCheckpoint(filepath="weights.keras", monitor="val_loss", save_best_only=True)

keras_callbacks = [es_callback, ck_callback]

# -----------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------
# Input  : fpre_train  (N_train, 9) — pre-collision populations
# Target : fpost_train (N_train, 9) — post-collision populations (BGK ground truth)
hist = model.fit(
    fpre_train, fpost_train,
    epochs=n_epochs,
    verbose=verbose,
    callbacks=keras_callbacks,
    validation_data=(fpre_test, fpost_test),
    batch_size=batch_size,
)

In [ ]:
# Reload the best checkpoint (in case restore_best_weights didn't catch it)
# and save the full model (architecture + weights) for use in a simulation.
model.load_weights("weights.keras")
model.save("example_network.keras")

In [ ]:
# Final RMSRE on the held-out test set.
# A value of ~1e-4 means the network's f_post is on average 0.01% off from the
# BGK reference — well within the accuracy needed for a stable LBM simulation.
model.evaluate(fpre_test, fpost_test)

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation loss on a log scale.
# Both curves should decrease together; a growing gap indicates overfitting.
plt.semilogy(hist.history['loss'],     lw=3, label='Training')
plt.semilogy(hist.history['val_loss'], lw=3, label='Validation')

plt.xlabel('Epoch')
plt.ylabel('RMSRE')
plt.legend(loc='best', frameon=False)
plt.show()